In [1]:
import ee
import pandas as pd

In [2]:
# 1. Authenticate and initialize GEE
# ==========================================
ee.Authenticate()
ee.Initialize(project="sundarbans-climate-analysis")

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


In [3]:
# 2. Study area: approximate coastal region
# west, south, east, north
# ==========================================
roi = ee.Geometry.Rectangle(
    [88.5, 20.5, 90.5, 22.0]
)

In [5]:
# 3. Load historical Copernicus salinity data
# ==========================================
collection = (
    ee.ImageCollection(
        "COPERNICUS/MARINE/GLOBAL_SEA_SURFACE/REP_DAILY"
    )
    .filterDate("2000-01-01", "2025-01-01")
    .select("sos")
)

print(
    "Available daily images:",
    collection.size().getInfo()
)

Available daily images: 9116


In [6]:
# 4. Create monthly date sequence
# ==========================================
start_date = ee.Date("2000-01-01")
month_count = 300  # Jan 2000 to Dec 2024

months = ee.List.sequence(0, month_count - 1)



In [7]:
# ==========================================
# 5. Calculate monthly mean salinity
# ==========================================
def calculate_monthly_salinity(index):
    month_start = start_date.advance(index, "month")
    month_end = month_start.advance(1, "month")

    monthly_images = collection.filterDate(
        month_start, month_end
    )

    image_count = monthly_images.size()

    monthly_mean = monthly_images.mean().rename("sos")

    stats = monthly_mean.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=roi,
        scale=14000,
        maxPixels=1e8,
        bestEffort=True
    )

    return ee.Feature(None, {
        "Year": month_start.get("year"),
        "Month": month_start.get("month"),
        "Date": month_start.format("YYYY-MM"),
        "Mean_Salinity_PSU": stats.get("sos"),
        "Available_Image_Count": image_count
    })

monthly_fc = ee.FeatureCollection(
    months.map(calculate_monthly_salinity)
)

# Remove months with no usable observations
monthly_fc = monthly_fc.filter(
    ee.Filter.gt("Available_Image_Count", 0)
).filter(
    ee.Filter.notNull(["Mean_Salinity_PSU"])
)


In [8]:
# 6. Transfer the monthly table to Python
# ==========================================
result = monthly_fc.getInfo()

rows = [
    feature["properties"]
    for feature in result["features"]
]

df = pd.DataFrame(rows)

df = df.sort_values(
    ["Year", "Month"]
).reset_index(drop=True)


In [9]:
# 7. Export to Excel
# ==========================================
output_file = "Sundarbans_Monthly_Salinity_2000_2024.xlsx"

with pd.ExcelWriter(
    output_file, engine="openpyxl"
) as writer:
    df.to_excel(
        writer,
        sheet_name="Monthly_Salinity",
        index=False
    )

print("\nExcel file created successfully!")
print("File:", output_file)
print("Monthly records:", len(df))
print(df.head(12).to_string(index=False))



Excel file created successfully!
File: Sundarbans_Monthly_Salinity_2000_2024.xlsx
Monthly records: 300
 Available_Image_Count    Date  Mean_Salinity_PSU  Month  Year
                    31 2000-01          29.923017      1  2000
                    29 2000-02          30.446180      2  2000
                    31 2000-03          31.185842      3  2000
                    30 2000-04          31.746893      4  2000
                    31 2000-05          31.972457      5  2000
                    30 2000-06          31.669591      6  2000
                    31 2000-07          30.855372      7  2000
                    31 2000-08          29.630122      8  2000
                    30 2000-09          28.654630      9  2000
                    31 2000-10          28.254020     10  2000
                    30 2000-11          28.670657     11  2000
                    31 2000-12          29.283752     12  2000
